# NanoReason training on Kaggle T4

A resumable three-session run: **baseline + SFT → GRPO → final evaluation + export**.

Before each run, enable **GPU T4 ×2** and **Internet** in notebook settings. NanoReason uses one T4 because Phi-3 fits on one 16 GB GPU and this TRL pipeline does not safely split across both cards.

## 1. Choose this session's stage

Run one stage per saved notebook version. For sessions 2 and 3, attach the preceding version's output using **Add Input → Your Work**.

In [1]:
RUN_STAGE = 'matched_eval'  # choose: 'sft', 'grpo', 'evaluate', 'matched_eval', or 'all'
assert RUN_STAGE in {'sft', 'grpo', 'evaluate', 'matched_eval', 'all'}


## 2. Get the code

In [2]:
import os, shutil
DST = '/kaggle/working/nanoreason'
if os.path.exists(DST):
    shutil.rmtree(DST)
clone_status = os.system('git clone --depth 1 https://github.com/pixeLatedbLue/NanoReason.git ' + DST)
if clone_status != 0:
    shutil.rmtree(DST, ignore_errors=True)
    raise RuntimeError(
        'Could not clone NanoReason. Earlier revisions fell back to an attached copy, '
        'which could silently substitute a different repository as the code directory '
        'and make the results unpairable; fix the clone instead.')
os.chdir(DST)
print('cwd =', os.getcwd())


Cloning into '/kaggle/working/nanoreason'...


cwd = /kaggle/working/nanoreason


## 3. Restore an earlier session, when attached

In [3]:
from pathlib import Path
def merge_tree(source: Path, destination: Path) -> None:
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True); print('Restored', source, 'to', destination)
input_root = Path('/kaggle/input'); artifact_roots = set()
for marker in list(input_root.glob('**/artifacts/**/adapter_config.json')) + list(input_root.glob('**/artifacts/grpo/checkpoint-*/trainer_state.json')):
    parts = marker.parts; artifact_roots.add(Path(*parts[:parts.index('artifacts') + 1]))
for artifact_root in sorted(artifact_roots):
    merge_tree(artifact_root, Path('/kaggle/working/artifacts'))
    merge_tree(artifact_root.parent / 'results', Path('/kaggle/working/results'))
evaluation_roots = {marker.parent for marker in input_root.glob('**/matched-eval/sample.json')}
for evaluation_root in sorted(evaluation_roots):
    merge_tree(evaluation_root, Path('/kaggle/working/matched-eval'))
if not evaluation_roots: print('No earlier matched-eval output attached; arms will not pair')
if not artifact_roots: print('No earlier NanoReason artifacts attached; starting without a restore')


Restored /kaggle/input/datasets/pixelatedblue/nanoreason-matched-eval-inputs/artifacts to /kaggle/working/artifacts
Restored /kaggle/input/datasets/pixelatedblue/nanoreason-matched-eval-inputs/matched-eval to /kaggle/working/matched-eval


## 4. Install and verify the environment

In [4]:
%env CUDA_VISIBLE_DEVICES=0
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!pip -q install "transformers>=4.46,<4.47" "trl==0.14.0" "peft>=0.12,<0.14" \
                "datasets>=2.21,<3.0" "accelerate>=0.34,<1.0" "bitsandbytes>=0.45.4,<0.51"
!pip -q install -e . --no-deps
!pip -q uninstall -y torchvision torchaudio
import torch
print('torch:', torch.__version__); print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'
!python -m unittest discover -s tests

env: CUDA_VISIBLE_DEVICES=0
env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0

## 5. Session 1 — baseline evaluation and SFT

In [5]:
from pathlib import Path
import subprocess
def run(command: list[str]) -> None:
    print('>', ' '.join(command)); subprocess.run(command, check=True)
if RUN_STAGE in {'sft', 'all'}:
    if not Path('/kaggle/working/results/baseline.json').exists():
        run(['python','-m','nanoreason.evaluate','--config','configs/kaggle_t4.toml','--run-name','baseline'])
    if not Path('/kaggle/working/artifacts/sft-final/adapter_config.json').exists():
        checkpoints=sorted(Path('/kaggle/working/artifacts/sft').glob('checkpoint-*'), key=lambda x:int(x.name.rsplit('-',1)[-1]))
        command=['python','-m','nanoreason.train_sft','--config','configs/kaggle_t4.toml']
        if checkpoints: command += ['--resume-from-checkpoint',str(checkpoints[-1])]
        run(command)
    else: print('SFT adapter already exists; skipping training')

## 6. Session 2 — GRPO

In [6]:
if RUN_STAGE in {'grpo', 'all'}:
    if not Path('/kaggle/working/artifacts/sft-final/adapter_config.json').exists():
        raise FileNotFoundError('Attach the Session 1 output before running GRPO')
    if not Path('/kaggle/working/artifacts/grpo-final/adapter_config.json').exists():
        checkpoints=sorted(Path('/kaggle/working/artifacts/grpo').glob('checkpoint-*'), key=lambda x:int(x.name.rsplit('-',1)[-1]))
        command=['python','-m','nanoreason.train_grpo','--config','configs/kaggle_t4.toml']
        if checkpoints: command += ['--resume-from-checkpoint',str(checkpoints[-1])]
        run(command)
    else: print('GRPO adapter already exists; skipping training')

## 7. Session 3 — evaluation, comparison, and export

In [7]:
if RUN_STAGE in {'evaluate', 'all'}:
    adapter='/kaggle/working/artifacts/grpo-final'
    if not Path(adapter,'adapter_config.json').exists(): raise FileNotFoundError('Attach the Session 2 output before evaluating')
    run(['python','-m','nanoreason.evaluate','--config','configs/kaggle_t4.toml','--adapter',adapter,'--run-name','grpo_final'])
    run(['python','-m','nanoreason.compare_results','--baseline','/kaggle/working/results/baseline.json','--candidate','/kaggle/working/results/grpo_final.json','--ci','--paired'])
    run(['python','-m','nanoreason.export','--config','configs/kaggle_t4.toml','--adapter',adapter,'--merged-dir','/kaggle/working/artifacts/merged','--load-4bit','--prompt','A pen costs 3 dollars. How much for 4 pens?'])

## 8. Session 4 — the full matched comparison, one arm per session

The n=64 comparison could not resolve the deltas it measured: 1.6 and 3.1 points against a sign test that needs roughly 10 at that size. This runs the same protocol over the whole 1319-question GSM8K test split, so a null result is a real null and a difference is a real difference.

Measured on an RTX 4060 the three arms cost 19.8 seconds per item together, about 7 hours for 1319 items. A T4 is roughly half that speed and a Kaggle session is capped at 12 hours, so set `MATCHED_ARMS` to one arm and run three sessions: `base`, then `sft`, then `grpo`.

Splitting them is safe because `environment_info` records library versions and the OS and nothing machine-specific, and the pinned installs above reproduce them. Two things do break pairing, both avoidable: editing `matched_eval.py`, `prompts.py`, `modeling.py` or `metrics.py` between sessions, since their SHA-256 is part of the recorded protocol; and changing the pinned versions.

Attach the previous session's output each time. The restore cell brings back `matched-eval/progress/`, so an arm cut short by the session limit continues instead of restarting. Run `base` first: the trained arms pair themselves against its `runs/base.json`.

This stage is not part of `'all'`: it needs the trained adapters attached, and it is long.

In [8]:
MATCHED_ARMS = 'sft'  # one of: 'base', 'sft', 'grpo'

SFT_SHA256 = '3cf54dcf0345452ff690bd71d90112979c67432f9f6275c1e77e6b722c5f6f73'
GRPO_SHA256 = '9cdc1fa17b4263e8de9899d36471f3fe005ed7f93cc4f3b86dabb17cc1554633'
EXPECTED = {'sft': SFT_SHA256, 'grpo': GRPO_SHA256}

if RUN_STAGE == 'matched_eval':
    needed = [a.strip() for a in MATCHED_ARMS.split(',') if a.strip() != 'base']
    command = ['python', 'experiments/run_matched_eval_all.py',
               '--arms', MATCHED_ARMS,
               '--samples', '1319',
               '--output', '/kaggle/working/matched-eval']
    for arm in needed:
        adapter = Path('/kaggle/working/artifacts', arm + '-final')
        if not (adapter / 'adapter_config.json').exists():
            raise FileNotFoundError(f'Attach the dataset holding {adapter}')
        if not list(adapter.glob('tokenizer*')):
            raise FileNotFoundError(
                f'{adapter} has no tokenizer files, which load_causal_lm reads from '
                'the adapter directory; copy them from the base model first')
        command += ['--' + arm + '-adapter', str(adapter),
                    '--' + arm + '-sha256', EXPECTED[arm]]
    if MATCHED_ARMS != 'base' and not Path('/kaggle/working/matched-eval/runs/base.json').exists():
        raise FileNotFoundError(
            'runs/base.json is not restored, so this arm could not be paired against '
            'the base arm; attach the dataset that carries matched-eval/')
    run(command)


> python experiments/run_matched_eval_all.py --arms sft --samples 1319 --output /kaggle/working/matched-eval --sft-adapter /kaggle/working/artifacts/sft-final --sft-sha256 3cf54dcf0345452ff690bd71d90112979c67432f9f6275c1e77e6b722c5f6f73
adapter verified: /kaggle/working/artifacts/sft-final/adapter_model.safetensors is 3cf54dcf0345452ff690bd71d90112979c67432f9f6275c1e77e6b722c5f6f73
fetching /kaggle/working/nanoreason/artifacts/slm/data/gsm8k-test.jsonl
Verified 1319 test items: /kaggle/working/nanoreason/artifacts/slm/data/gsm8k-test.jsonl
SHA256: 3730d312f6e3440559ace48831e51066acaca737f6eabec99bccb9e4b3c39d14

=== sft ===
> /usr/bin/python3 -m nanoreason.matched_eval --arm sft --config configs/local_inference.toml --source artifacts/slm/data/gsm8k-test.jsonl --output /kaggle/working/matched-eval --samples 1319 --seed 42 --max-new-tokens 512 --adapter /kaggle/working/artifacts/sft-final


I0000 00:00:1793713544.596571     233 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loading sft; resuming at 0/1319


Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.31s/it]


sft: 1/1319, correct=0
sft: 2/1319, correct=1
sft: 3/1319, correct=2
sft: 4/1319, correct=3
sft: 5/1319, correct=4
sft: 6/1319, correct=5
sft: 7/1319, correct=6
sft: 8/1319, correct=6
sft: 9/1319, correct=7
sft: 10/1319, correct=8
sft: 11/1319, correct=9
sft: 12/1319, correct=10
sft: 13/1319, correct=11
sft: 14/1319, correct=12
sft: 15/1319, correct=13
sft: 16/1319, correct=14
sft: 17/1319, correct=15
sft: 18/1319, correct=16
sft: 19/1319, correct=17
sft: 20/1319, correct=17
sft: 21/1319, correct=18
sft: 22/1319, correct=18
sft: 23/1319, correct=19
sft: 24/1319, correct=20
sft: 25/1319, correct=21
sft: 26/1319, correct=22
sft: 27/1319, correct=23
sft: 28/1319, correct=24
sft: 29/1319, correct=25
sft: 30/1319, correct=26
sft: 31/1319, correct=27
sft: 32/1319, correct=28
sft: 33/1319, correct=29
sft: 34/1319, correct=30
sft: 35/1319, correct=30
sft: 36/1319, correct=31
sft: 37/1319, correct=32
sft: 38/1319, correct=33
sft: 39/1319, correct=34
sft: 40/1319, correct=35
sft: 41/1319, correc

## 9. Save and continue

After a stage completes, save a notebook version and attach that version's output to the next session. The notebook restores the latest checkpoints automatically.

Expected outputs: `results/baseline.json`, `artifacts/sft-final/`, `artifacts/grpo-final/`, `results/grpo_final.json`, `results/forgetting_probe.json`, and `artifacts/merged/`.

The matched comparison stage writes `matched-eval/` instead: one `runs/*.json` per arm, plus `base-vs-sft.json` and `base-vs-grpo.json`.
